In [16]:
import sys
from pathlib import Path

BASE = Path("/workspace/tiny-llm-from-scratch")
sys.path.append(str(BASE))

print("Project Root :", BASE)

Project Root : /workspace/tiny-llm-from-scratch


In [17]:
import json
import random

import torch

from tokenizer.char_tokenizer import CharTokenizer
from model.tiny_transformer import TinyGPT, GPTConfig

In [18]:
CHECKPOINT = BASE / "checkpoints/tinyllm_100k_char_best.pt"

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu"
)

print(checkpoint.keys())

dict_keys(['model_state', 'optimizer_state', 'config', 'gpt_config', 'tokenizer_path', 'step', 'best_val_loss', 'param_count'])


In [19]:
tokenizer = CharTokenizer.load(
    BASE / "tokenizer/tokenizer_char.json"
)

print("Vocabulary Size :", tokenizer.vocab_size)

Vocabulary Size : 44


In [20]:
cfg = GPTConfig(**checkpoint["gpt_config"])

model = TinyGPT(cfg)

model.load_state_dict(
    checkpoint["model_state"]
)

model.eval()

print("Model Loaded Successfully")

Model Loaded Successfully


In [21]:
print("="*60)
print("MODEL SUMMARY")
print("="*60)

print("Model Name      :", checkpoint["config"]["model_name"])
print("Parameters      :", checkpoint["param_count"])
print("Vocabulary Size :", tokenizer.vocab_size)
print("Layers          :", cfg.n_layer)
print("Heads           :", cfg.n_head)
print("Embedding       :", cfg.n_embd)
print("Block Size      :", cfg.block_size)
print("Dropout         :", cfg.dropout)

MODEL SUMMARY
Model Name      : tinyllm_100k_char
Parameters      : 111104
Vocabulary Size : 44
Layers          : 2
Heads           : 4
Embedding       : 64
Block Size      : 128
Dropout         : 0.1


In [22]:
print("="*60)

print("CHECKPOINT")

print("="*60)

print("Training Step :", checkpoint["step"])
print("Best Loss     :", checkpoint["best_val_loss"])
print("Device        :", checkpoint["config"]["device"])
print("Learning Rate :", checkpoint["config"]["learning_rate"])
print("Batch Size    :", checkpoint["config"]["batch_size"])
print("Max Steps     :", checkpoint["config"]["max_steps"])

CHECKPOINT
Training Step : 4750
Best Loss     : 0.05055588111281395
Device        : cpu
Learning Rate : 0.0003
Batch Size    : 16
Max Steps     : 5000


In [23]:
print("="*60)
print("VOCABULARY")
print("="*60)

print("Vocabulary Size :", tokenizer.vocab_size)

print()

print("{:<5} {}".format("ID","TOKEN"))
print("-"*30)

for idx in sorted(tokenizer.itos):

    print("{:<5} {}".format(
        idx,
        repr(tokenizer.itos[idx])
    ))

VOCABULARY
Vocabulary Size : 44

ID    TOKEN
------------------------------
0     '<UNK>'
1     '\n'
2     ' '
3     ','
4     '-'
5     '.'
6     'A'
7     'D'
8     'F'
9     'G'
10    'I'
11    'L'
12    'M'
13    'P'
14    'Q'
15    'R'
16    'S'
17    'T'
18    'a'
19    'b'
20    'c'
21    'd'
22    'e'
23    'f'
24    'g'
25    'h'
26    'i'
27    'j'
28    'k'
29    'l'
30    'm'
31    'n'
32    'o'
33    'p'
34    'q'
35    'r'
36    's'
37    't'
38    'u'
39    'v'
40    'w'
41    'x'
42    'y'
43    'z'


In [24]:
samples=[

"Python",

"Docker",

"Machine",

"SQL",

"AI"

]

for text in samples:

    ids=tokenizer.encode(text)

    decoded=tokenizer.decode(ids)

    print("="*40)

    print("Input   :",text)

    print("Encoded :",ids)

    print("Decoded :",decoded)

Input   : Python
Encoded : [13, 42, 37, 25, 32, 31]
Decoded : Python
Input   : Docker
Encoded : [7, 32, 20, 28, 22, 35]
Decoded : Docker
Input   : Machine
Encoded : [12, 18, 20, 25, 26, 31, 22]
Decoded : Machine
Input   : SQL
Encoded : [16, 14, 11]
Decoded : SQL
Input   : AI
Encoded : [6, 10]
Decoded : AI


In [25]:
samples=[

"Hello",

"HELLO",

"Python",

"SQL",

"HTML",

"Assalamu Alaikum",

"السلام عليكم",

"12345",

"!@#$"

]

for text in samples:

    print("="*60)

    print(text)

    for ch in text:

        if ch in tokenizer.stoi:

            print("✓",repr(ch))

        else:

            print("✗",repr(ch),"UNKNOWN")

Hello
✗ 'H' UNKNOWN
✓ 'e'
✓ 'l'
✓ 'l'
✓ 'o'
HELLO
✗ 'H' UNKNOWN
✗ 'E' UNKNOWN
✓ 'L'
✓ 'L'
✗ 'O' UNKNOWN
Python
✓ 'P'
✓ 'y'
✓ 't'
✓ 'h'
✓ 'o'
✓ 'n'
SQL
✓ 'S'
✓ 'Q'
✓ 'L'
HTML
✗ 'H' UNKNOWN
✓ 'T'
✓ 'M'
✓ 'L'
Assalamu Alaikum
✓ 'A'
✓ 's'
✓ 's'
✓ 'a'
✓ 'l'
✓ 'a'
✓ 'm'
✓ 'u'
✓ ' '
✓ 'A'
✓ 'l'
✓ 'a'
✓ 'i'
✓ 'k'
✓ 'u'
✓ 'm'
السلام عليكم
✗ 'ا' UNKNOWN
✗ 'ل' UNKNOWN
✗ 'س' UNKNOWN
✗ 'ل' UNKNOWN
✗ 'ا' UNKNOWN
✗ 'م' UNKNOWN
✓ ' '
✗ 'ع' UNKNOWN
✗ 'ل' UNKNOWN
✗ 'ي' UNKNOWN
✗ 'ك' UNKNOWN
✗ 'م' UNKNOWN
12345
✗ '1' UNKNOWN
✗ '2' UNKNOWN
✗ '3' UNKNOWN
✗ '4' UNKNOWN
✗ '5' UNKNOWN
!@#$
✗ '!' UNKNOWN
✗ '@' UNKNOWN
✗ '#' UNKNOWN
✗ '$' UNKNOWN


In [26]:
params=sum(

p.numel()

for p in model.parameters()

)

size=sum(

p.numel()*p.element_size()

for p in model.parameters()

)/(1024*1024)

print("Parameters :",params)

print(f"Approx Size : {size:.2f} MB")

Parameters : 111104
Approx Size : 0.42 MB


In [28]:
def generate(

prompt,

temperature=0.8,

top_k=20,

tokens=120

):

    ids=tokenizer.encode(prompt)

    x=torch.tensor([ids])

    with torch.inference_mode():

        output=model.generate(

            x,

            max_new_tokens=tokens,

            temperature=temperature,

            top_k=top_k

        )

    return tokenizer.decode(

        output[0].tolist()

    )

In [29]:
prompts=[

"Python",

"Docker",

"SQL",

"Machine Learning",

"Artificial Intelligence",

"Transformer",

"FastAPI",

"Database",

"Assalamu Alaikum"

]

for prompt in prompts:

    print("="*70)

    print(generate(prompt))

Python web framework used to build web applications.
FastAPI is a Python framework used to build APIs.
Docker is used to packa
Docker is used to build web applications.
FastAPI is a Python framework used to build APIs.
Docker is used to package applicat
SQL is a relational database.
RAG means Retrieval Augmented Generation.
RAG helps a language model answer using external do
Machine Learning models improve by reducing loss during trainining.
The training loop includes forward pass, loss calculation, backpropa
Artificial Intelligence is the science of making machines learn from data.
A language model learns to predict the next token in a sequence.
A t
Transformer is a neural network architecture used in modern language models.
Django is a Python web framework used to build web app
FastAPI is a Python framework used to build APIs.
Docker is used to package applications and run them inside containers.
A data
Database stores structured information.
PostgreSQL is a relational database.
RAG 

In [30]:
for temp in [

0.2,

0.5,

0.8,

1.0,

1.2

]:

    print("="*70)

    print("Temperature :",temp)

    print(generate(

        "Python",

        temperature=temp

    ))

Temperature : 0.2
Python web framework used to build web applications.
FastAPI is a Python framework used to build APIs.
Docker is used to packa
Temperature : 0.5
Python web framework used to build web applications.
FastAPI is a Python framework used to build APIs.
Docker is used to packa
Temperature : 0.8
Python web framework used to build web applications.
FastAPI is a Python framework used to build APIs.
Docker is used to packa
Temperature : 1.0
Python web framework used to build web applications.
FastAPI is a Python framework used to build APIs.
Docker is used to packa
Temperature : 1.2
Python web framework used to build web appplications.
FastAPI is a Python framework used to build APIs.
Docker is used to pack


In [31]:
for k in [

5,

10,

20,

40

]:

    print("="*70)

    print("Top K :",k)

    print(generate(

        "Docker",

        top_k=k

    ))

Top K : 5
Docker is used to package applications and run them inside containers.
A database stores structured information.
PostgreSQL is
Top K : 10
Docker is used to package applications and run them inside containers.
A database stores structured information.
PostgreSQL is
Top K : 20
Docker is used to package applications and run them inside containers.
A database stores structured information.
PostgreSQL is
Top K : 40
Docker is used to package applications and run them inside containers.
A database stores structured information.
PostgreSQL is


In [32]:
for i in range(5):

    print("="*60)

    print(generate(

        "Python",

        temperature=0.2

    ))

Python web framework used to build web applications.
FastAPI is a Python framework used to build APIs.
Docker is used to packa
Python web framework used to build web applications.
FastAPI is a Python framework used to build APIs.
Docker is used to packa
Python web framework used to build web applications.
FastAPI is a Python framework used to build APIs.
Docker is used to packa
Python web framework used to build web applications.
FastAPI is a Python framework used to build APIs.
Docker is used to packa
Python web framework used to build web applications.
FastAPI is a Python framework used to build APIs.
Docker is used to packa


In [33]:
print("="*60)

print("Checkpoint Keys")

print("="*60)

for key,value in checkpoint.items():

    print(key,type(value))

Checkpoint Keys
model_state <class 'collections.OrderedDict'>
optimizer_state <class 'dict'>
config <class 'dict'>
gpt_config <class 'dict'>
tokenizer_path <class 'str'>
step <class 'int'>
best_val_loss <class 'float'>
param_count <class 'int'>


In [34]:
report=[]

report.append("TinyGPT Evaluation Report")

report.append("="*60)

report.append(f"Model : {checkpoint['config']['model_name']}")

report.append(f"Parameters : {params}")

report.append(f"Vocabulary : {tokenizer.vocab_size}")

report.append(f"Layers : {cfg.n_layer}")

report.append(f"Heads : {cfg.n_head}")

report.append(f"Embedding : {cfg.n_embd}")

report.append(f"Block Size : {cfg.block_size}")

report.append(f"Training Step : {checkpoint['step']}")

report.append(f"Best Loss : {checkpoint['best_val_loss']}")

REPORT=BASE/"reports/evaluation_report.txt"

REPORT.write_text(

"\n".join(report),

encoding="utf-8"

)

print("Saved :",REPORT)

Saved : /workspace/tiny-llm-from-scratch/reports/evaluation_report.txt


In [35]:
print("="*60)

print("TinyGPT Evaluation Completed")

print("="*60)

print("✓ Checkpoint Loaded")

print("✓ Tokenizer Loaded")

print("✓ Vocabulary Verified")

print("✓ Model Restored")

print("✓ Generation Working")

print("✓ Temperature Tested")

print("✓ Top-K Tested")

print("✓ Report Saved")

print("="*60)

TinyGPT Evaluation Completed
✓ Checkpoint Loaded
✓ Tokenizer Loaded
✓ Vocabulary Verified
✓ Model Restored
✓ Generation Working
✓ Temperature Tested
✓ Top-K Tested
✓ Report Saved
